# Scraping: récupération du corpus CAMILLE de la page ULB du professeur Max De Wild (téléchargement automatique de 51 fichiers PDF)

Dans ce notebook, je crée un robot qui va ouvrir la page de M. Max De Wilde et récupérer un sous-ensemble du corpus CAMILLE [page ulb](https://max.de.wilde.web.ulb.be/camille/) et récupérer tous les PDFs du corpus et les stocker dans un fichier csv.

## Imports

In [2]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Récupération des liens PDF de la page CAMILLE



In [3]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

pdfs=[] #j'ai créé une liste qui contiendra les fichiers PDFs

root_url = f"https://max.de.wilde.web.ulb.be/camille/"

response = requests.get(root_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

for link in soup.find_all("a"):
    url = link.get("href") #ici je cherche les liens pointant vers les PDF du corpus et non pas le script HTML des articles comme on a fait avec RTBF
    if url and url.lower().endswith(".pdf"): #car j'ai vu que les liens listés sur la page de M. De Wilde (non seulement sont écrit en miniscule, mais aussi..) finissent tous en .pdf, donc un bon repère pour dire que ce qui finit en .pdf est un lien pointant vers un document pdf
        # Récupération du nom du fichier
        filename = url.split("/")[-1] #les mêmes liens contiennet les noms des documents auxquels ils pointent (chaque PDF dans le sous-ensemble contient un nom, j'exploite ça)

        # Construction de l'URL complet
        pdf_url = requests.compat.urljoin(root_url, url)

        pdfs.append([pdf_url, filename])

In [4]:
# Affichage du nombre de PDFs récupérés
len(pdfs)

51

In [5]:
# Affichage des 10 premières entrées (donc les 10 premiers PDFs téléchargés)
pdfs[:10]

[['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1892-08-07_01-0003.pdf',
  'KB_JB230_1892-08-07_01-0003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB427_1920-01-10_01-00004.pdf',
  'KB_JB427_1920-01-10_01-00004.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB555_1836-02-08_01-00002.pdf',
  'KB_JB555_1836-02-08_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB638_1860-05-21_01-00002.pdf',
  'KB_JB638_1860-05-21_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB773_1918-11-30_01-00002.pdf',
  'KB_JB773_1918-11-30_01-00002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB838_1887-12-28_01-00003.pdf',
  'KB_JB838_1887-12-28_01-00003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB230_1903-10-16_01-0002.pdf',
  'KB_JB230_1903-10-16_01-0002.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB427_1933-01-04_01-00003.pdf',
  'KB_JB427_1933-01-04_01-00003.pdf'],
 ['https://max.de.wilde.web.ulb.be/camille/KB_JB555_1899-01-19_01-00

## Création d'un dataframe avec les liens et les titres des articles


In [6]:
df = pd.DataFrame(pdfs, columns=['link', 'filename'])
df

,link,filename
0,https://max.de.wilde.web.ulb.be/camille/KB_JB2...,KB_JB230_1892-08-07_01-0003.pdf
1,https://max.de.wilde.web.ulb.be/camille/KB_JB4...,KB_JB427_1920-01-10_01-00004.pdf
2,https://max.de.wilde.web.ulb.be/camille/KB_JB5...,KB_JB555_1836-02-08_01-00002.pdf
3,https://max.de.wilde.web.ulb.be/camille/KB_JB6...,KB_JB638_1860-05-21_01-00002.pdf
4,https://max.de.wilde.web.ulb.be/camille/KB_JB7...,KB_JB773_1918-11-30_01-00002.pdf
5,https://max.de.wilde.web.ulb.be/camille/KB_JB8...,KB_JB838_1887-12-28_01-00003.pdf
6,https://max.de.wilde.web.ulb.be/camille/KB_JB2...,KB_JB230_1903-10-16_01-0002.pdf
7,https://max.de.wilde.web.ulb.be/camille/KB_JB4...,KB_JB427_1933-01-04_01-00003.pdf
8,https://max.de.wilde.web.ulb.be/camille/KB_JB5...,KB_JB555_1899-01-19_01-00003.pdf
9,https://max.de.wilde.web.ulb.be/camille/KB_JB6...,KB_JB638_1902-12-20_01-00002.pdf


In [11]:
# Création du dossier là ou on va mettre les données (le tableau CSV)
os.makedirs("../PDFsCamille", exist_ok=True)

# Sauvegarde du dataframe dans un fichier csv
df.to_csv(f"../PDFsCamille_{time.strftime('%Y%m%d')}.csv", index=False)

## Téléchargement automatique des 51 fichiers PDFs

##premier essai avec le premier document:

In [15]:
import os
# Création du dossier destiné à recevoir les fichiers PDF
os.makedirs("../PDFsCamille", exist_ok=True)

# Récupération du premier PDF du corpus Camille
pdf_url = df.iloc[0,0]

# Récupération du nom du fichier
filename = pdf_url.split("/")[-1]

# Ouverture de la page
response = requests.get(pdf_url, headers=headers)

# vérification de la réponse
response.raise_for_status()

# Création du fichier PDF
with open(f"../PDFsCamille/{filename}", "wb") as writer:
    writer.write(response.content)

print("Téléchargement terminé !")

Téléchargement terminé !


puisque c'est réussi, on télécharge le reste des documents (les pdfs du corpus)

In [17]:
# maintenant que la célule précédente a bien fonctionné, je télécharge le reste du corpus
for i in range(len(df)): #déjà j'introduit i, ensuite je l'utilise pour dire "pour chaque document "i" de la liste pdf_url, fais ce que je te demanderai de faire dans les lignes suivantes
    pdf_url = df.iloc[i,0] #pour chaque ligne "i", mais uniquement la première colonne de notre tableau (donc les liens que nous avons déjà préparé)

    # Récupération du nom du fichier
    filename = pdf_url.split("/")[-1]

    # Ouverture de la page
    response = requests.get(pdf_url, headers=headers)

    # vérification de la réponse
    response.raise_for_status()

    # Création du fichier PDF
    with open(f"../PDFsCamille/{filename}", "wb") as writer:
        writer.write(response.content)

print("Téléchargement terminé pour le reste des PDFs :D !")

Téléchargement terminé pour le reste des PDFs :D !


### Nettoyage du texte à l'aide d'expressions régulières

 le téléchargement des 51 PDF ne requiert pas de regex (regular expressions) pour le nettoyage. Car cet outil est exploitable lors de l'extraction de text et est applicable aux fichiers .txt (donc elle n'est pas exploitable dans notre cas puisque notre objectif est de télécharger des documents PDFs uniquement)

 Sauf si on extrait le text des PDFs qu'on vient de télécharger (en utilisant des bibliothèques comme pypdf et l'outil pdfreader) pour convertir les pdf en text (on peut rencontrer des problèmes d'interprétabilité vu que le corpus CAMILLE est OCR-isé, et est difficile à lire vu la qualité du scan, la qualité du papier/ l'impression (vu l vieillesse du papier), etc..), et onsuite on applique la même opération de nettoyage du text comme on a fait avec les texts extraits du site RTBF 

 Je vais quand même faire un essaie de pratique avec le premier document PDF du corpus Camille: je vais donc suivre les étapes suivantes:
 1. importer l'outil PdfReader
 2. Je vais selectionner le lien du PDF que je souhaite travailler dessus (ce sera PDF N°1 dans ce cas) de la même liste que j'ai créé toute à l'heure
 3. Je vais ensuite télécharger le document 
 4. c'est l'étape de la lecture te l'extraction du text
 5. maintenant c'ets le nettoyage avec les RegEx
 6. finalement je sauvegarde le pdf et le text extrait et nettoyé dans un document text (.txt)

 Nihal SAKHRI :D 

In [20]:
from pypdf import PdfReader

# 1. Lien du PDF
pdf_url = df.iloc[0, 0] #(j'aurais dû faire la même opération en boucle pour couvrir tous les documents en utilisant une boucle for)

# 2. Téléchargement du document PDF
response = requests.get(pdf_url, headers=headers, timeout=60)
response.raise_for_status()

os.makedirs("../PDFsCamille", exist_ok=True)
pdf_path = "../PDFsCamille/document_1.pdf"

with open(pdf_path, "wb") as f:
    f.write(response.content)

# 3. Extraction du texte
reader = PdfReader(pdf_path)
content = "\n".join(page.extract_text() or "" for page in reader.pages)

# 4. Nettoyage avec expressions régulières
clean_content = re.sub(r"\([^\)]+\)", " ", content)
clean_content = re.sub(r"\s+", " ", clean_content).strip()

# 5. Sauvegarde du PDF et du texte nettoyé en fichier .txt
txt_path = "../PDFsCamille/document_1.txt"

with open(txt_path, "w", encoding="utf-8") as writer:
    writer.write(clean_content)

print("PDF enregistré :", pdf_path)
print("Texte nettoyé enregistré :", txt_path)


PDF enregistré : ../PDFsCamille/document_1.pdf
Texte nettoyé enregistré : ../PDFsCamille/document_1.txt


après consultation du document résultant de l'extraction et le fichier text après nettoyage, on confirme qu'il n'est pas lisible par la machine (ni par moi-même, d'ailleurs), vu la qualité de l'OCR-isation, la qualité du papier (et son âge), la qualité de l'impression etc..

### Travail fait par Nihal SAKHRI :)
